# BiDirectional LSTM + 1D Conv — Best Model ⭐

**Challenge:** AN2DL 2024 — Pirate Pain Classification  
**Approach:** Feature-wise 1D convolution to extract local patterns from 30 joint-angle channels,  
followed by a Bidirectional LSTM that captures forward and backward temporal dependencies.  
Categorical morphology features (`n_legs`) are embedded via `nn.Embedding`.  
**Result:** val F1 ≈ **0.935** — best among all attempted architectures.

### Why this works best
- **Conv1D (feature-wise):** `in_channels=30 joints → out_channels=16` extracts local cross-joint patterns at each timestep
- **BiDir LSTM:** sees both past and future timesteps within each window
- **Embedding:** treats the pirate's morphology as a learned representation rather than a raw integer
- **Weighted CE loss:** compensates for the 77%/14%/8% class imbalance (no_pain / low_pain / high_pain)
- **Majority voting:** final label per pirate is the mode across all sliding windows of that sequence

### Architecture summary
```
Input branches:
  Joint angles (30) ──► Conv1D(30→16, k=3, same-padding) ──►
  n_legs integer    ──► Embedding(vocab=2, dim=3)          ──► Concat(16+3+4=23)
  Pain surveys (4)  ─────────────────────────────────────────►
                         BiLSTM(hidden=128, layers=2, dropout=0.2)
                              ↓ last hidden state (fwd ‖ bwd = 256)
                         Linear(256 → 3) + LogSoftmax
```

### Hyperparameters
| Parameter | Value |
|-----------|-------|
| Window size | 20 |
| Stride | 5 |
| Hidden size | 128 |
| LSTM layers | 2 |
| Dropout | 0.2 |
| Learning rate | 1e-3 |
| Batch size | 512 |
| Max epochs | 512 |
| Early stopping patience | 50 |

In [ ]:
#Libraries import

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt # Plotting 
import seaborn as sns # Statistical data visualization
import warnings # To suppress warnings
import torch 
from torch import nn #Neural Networks 
from torch.utils.data import TensorDataset, DataLoader
import random
import logging   
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix  # Usefull metrics
from sklearn.model_selection import train_test_split
import re           # Regex
from sklearn.decomposition import PCA
import math

In [ ]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
SEED = 42 # To guarantee reproducibility
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Set torch options

logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models


# Use a GPU if available, or use a CPU instead (N.B. GPUs are made available from kaggle)
if torch.cuda.is_available():     
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

# Check torch version 
print(f"PyTorch version: {torch.__version__}")


# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

In [ ]:
#Set environment variables

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

In [ ]:
#Suppress warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Data Loading

In [ ]:
#Data Loading

X = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_train.csv')
y = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_train_labels.csv')
X_test = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_test.csv')

## Data Exploration and Analysis

Key observations:
- **661 training samples**, **1324 test samples**, each with **160 timesteps × 40 features**
- `joint_30` has zero variance (constant 0.5) → dropped
- Classes: `no_pain` (77 %), `low_pain` (14 %), `high_pain` (8 %) — strongly imbalanced
- Categorical features: `n_legs` (two / one+peg_leg), `n_hands`, `n_eyes` — treated as embeddings
- Joints 21-25 are near-zero / sparse (different measurement unit or sparse events)

In [ ]:
# dropping column with 0 variance
X.drop('joint_30', axis=1, inplace=True)
X_test.drop('joint_30', axis=1, inplace=True)

In [ ]:
# How many samples?
samples = X['sample_index'].unique()
n_samples = len(samples) 
print(n_samples) # 661

# Data Preprocessing

In [ ]:
label_map = {
    'no_pain' : 0,
    'low_pain' : 1,
    'high_pain' : 2,
    0 : 0,
    1 : 1,
    2 : 2
}

y['label'] = y['label'].map(label_map)

In [ ]:
sample_labels = []

for sample in samples:
    sample_labels.append([sample, int(y[y['sample_index']==sample]['label'].unique())])

In [ ]:
labels_df = pd.DataFrame(sample_labels)
print(labels_df)

In [ ]:
train_samples, val_samples = train_test_split(labels_df, test_size = 60, random_state = SEED, shuffle=True, stratify=labels_df[1])

In [ ]:
train_samples = train_samples[0]
val_samples = val_samples[0]

In [ ]:
emb_cols = ['sample_index', 'time', 'n_legs']   # 'n_hands', 'n_eyes' missing for now

joint_cols = [col for col in X.columns if re.match(r'joint_\d{2}', col)]
joint_cols.append('sample_index')
joint_cols.append('time')

pain_cols = [col for col in X.columns if re.match(r'pain_survey.*', col)]
pain_cols.append('sample_index')
pain_cols.append('time')

In [ ]:
X_emb = X[emb_cols]
X_test_emb = X_test[emb_cols]

In [ ]:
X_joint = X[joint_cols]
X_test_joint = X_test[joint_cols]

In [ ]:
X_pain = X[pain_cols]
X_test_pain = X_test[pain_cols]

In [ ]:
# Split the dataset into training and validation sets based on samples ids
X_train_emb = X_emb[X_emb['sample_index'].isin(train_samples)]
X_val_emb = X_emb[X_emb['sample_index'].isin(val_samples)]
X_train_joint = X_joint[X_joint['sample_index'].isin(train_samples)]
X_val_joint = X_joint[X_joint['sample_index'].isin(val_samples)]
X_train_pain = X_pain[X_pain['sample_index'].isin(train_samples)]
X_val_pain = X_pain[X_pain['sample_index'].isin(val_samples)]

In [ ]:
y_train = y[y['sample_index'].isin(train_samples)]
y_val = y[y['sample_index'].isin(val_samples)]

In [ ]:
# Study labels distribution in training set 

training_labels = {
    0 : 0,
    1 : 0,
    2 : 0
}


# Count occurrences of each activity for unique IDs in the training set
for id in y_train['sample_index'].unique():
    label = y_train[y_train['sample_index'] == id]['label'].values[0]
    training_labels[label] += 1

# Print the distribution of training labels
print('Training labels:', training_labels)
print('Training distributions:\n')
for label in training_labels:
    stat = training_labels[label]/ (training_labels[0] + training_labels[1] + training_labels[2])
    print(f'{label}: {stat}')

In [ ]:
# Study labels distribution in validation set 

validation_labels = {
    0 : 0,
    1 : 0,
    2 : 0
}


# Count occurrences of each activity for unique IDs in the validation set
for id in y_val['sample_index'].unique():
    label = y_val[y_val['sample_index'] == id]['label'].values[0]
    validation_labels[label] += 1


# Print the distribution of validation labels
print('Validation labels:', validation_labels)
print('Validation distributions:\n')
for label in validation_labels:
    stat = validation_labels[label]/ (validation_labels[0] + validation_labels[1] + validation_labels[2])
    print(f'{label}: {stat}')

In [ ]:
X_train_emb['n_legs'] = X_train_emb['n_legs'].replace('two', 'two_legs')

X_val_emb['n_legs'] = X_val_emb['n_legs'].replace('two', 'two_legs')

X_test_emb['n_legs'] = X_test_emb['n_legs'].replace('two', 'two_legs')

In [ ]:
#Preparation for embedding
embedding_map = {
    'two_legs' : 0,
    'one+peg_leg' : 1
}

In [ ]:
X_train_emb['n_legs'] = X_train_emb['n_legs'].map(embedding_map)

In [ ]:
X_val_emb['n_legs'] = X_val_emb['n_legs'].map(embedding_map)

In [ ]:
X_test_emb['n_legs'] = X_test_emb['n_legs'].map(embedding_map)

In [ ]:
joint_cols.remove('sample_index')
joint_cols.remove('time')

In [ ]:
#Normalization with minMax
mins = X_train_joint[joint_cols].min()
maxs = X_train_joint[joint_cols].max()

for col in joint_cols:
    den = maxs[col] - mins[col]
    
    if den == 0:
        den = maxs[col]
        
    X_train_joint[col] = (X_train_joint[col] - mins[col])/den # Normalize train set
    X_val_joint[col] = (X_val_joint[col] - mins[col])/den # Normalize val set
    X_test_joint[col] = (X_test_joint[col] - mins[col])/den # Normalize test set

In [ ]:
training_labels = {
    0 : 0,
    1 : 0,
    2 : 0,
}


# Count occurrences of each activity for unique IDs in the training set
for id in y_train['sample_index'].unique():
    label = y_train[y_train['sample_index'] == id]['label'].values[0]
    training_labels[label] += 1

class_counts = torch.tensor([training_labels[0], training_labels[1], training_labels[2]], dtype=torch.float)

In [ ]:
emb_cols.remove('sample_index')
emb_cols.remove('time')

pain_cols.remove('sample_index')
pain_cols.remove('time')

In [ ]:
y['label'] = y['label'].map(label_map)

In [ ]:
 # Window size and stride for sequence building

WINDOW_SIZE =20
STRIDE = 5 # the window need to be divisible by the stride

In [ ]:
# Function to build sequences

def build_sequences(df, window, stride, isTest, data_cols):
    # Be sure that the window is divisible by the stride
    assert window%stride == 0

    #Lists to store sequences and corresponding labels
    dataset = []
    labels = []

    # Iterate over the ids
    for id in df['sample_index'].unique():
        id_int = int(id)
        # Extract data from the current id
        temp = df[df['sample_index'] == id_int][data_cols].values

        if(not isTest):
            # Retrive the label for the id
            label = y[y['sample_index'] == id_int]['label'].values[0]

        # Build feature windows and associate them with labels
        idx = 0
        while idx + window <= len(temp):
            dataset.append(temp[idx:idx + window])
            if(not isTest):
                labels.append(label)
            idx += stride

    # Convert lists to numpy arrays for further processing
    dataset = np.array(dataset)
    if(not isTest):
        labels = np.array(labels)

    if(not isTest):
        return dataset, labels
    else:
        return dataset


In [ ]:
X_train_seq_emb, y_train_seq_emb = build_sequences(X_train_emb, WINDOW_SIZE, STRIDE, False, emb_cols)
X_val_seq_emb, y_val_seq_emb = build_sequences(X_val_emb, WINDOW_SIZE, STRIDE, False, emb_cols)
X_test_seq_emb = build_sequences(X_test_emb, WINDOW_SIZE, STRIDE, True, emb_cols)

In [ ]:
# Define the number of classes based on the categorical labels
num_classes = len(np.unique(y_train_seq_emb))

In [ ]:
X_train_seq_joint, y_train_seq_joint = build_sequences(X_train_joint, WINDOW_SIZE, STRIDE, False, joint_cols)
X_val_seq_joint, y_val_seq_joint = build_sequences(X_val_joint, WINDOW_SIZE, STRIDE, False, joint_cols)
X_test_seq_joint = build_sequences(X_test_joint, WINDOW_SIZE, STRIDE, True, joint_cols)

In [ ]:
X_train_seq_pain, y_train_seq_pain = build_sequences(X_train_pain, WINDOW_SIZE, STRIDE, False, pain_cols)
X_val_seq_pain, y_val_seq_pain = build_sequences(X_val_pain, WINDOW_SIZE, STRIDE, False, pain_cols)
X_test_seq_pain = build_sequences(X_test_pain, WINDOW_SIZE, STRIDE, True, pain_cols)

In [ ]:
train_ds_cat = TensorDataset(torch.from_numpy(X_train_seq_emb), torch.from_numpy(X_train_seq_joint), torch.from_numpy(X_train_seq_pain) ,torch.from_numpy(y_train_seq_emb))
val_ds_cat   = TensorDataset(torch.from_numpy(X_val_seq_emb), torch.from_numpy(X_val_seq_joint), torch.from_numpy(X_val_seq_pain) ,torch.from_numpy(y_val_seq_emb))
test_ds_cat = TensorDataset(torch.from_numpy(X_test_seq_emb), torch.from_numpy(X_test_seq_joint), torch.from_numpy(X_test_seq_pain))

In [ ]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [ ]:
BATCH_SIZE = 512

In [ ]:
train_cat_loader = make_loader(train_ds_cat, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_cat_loader   = make_loader(val_ds_cat, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_cat_loader   = make_loader(test_ds_cat, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# Network parameters

In [ ]:
# Training configuration
LEARNING_RATE = 1e-3
EPOCHS = 512
PATIENCE = 50

# Architecture
HIDDEN_LAYERS =  2       # Hidden layers
HIDDEN_SIZE = 128        # Neurons per layer

# Regularisation
DROPOUT_RATE = 0.2         # Dropout probability
L1_LAMBDA = 1e-3            # L1 penalty
L2_LAMBDA = 1e-3        # L2 penalty

In [ ]:
weights = 1/class_counts  # To give more importance to misclassifications of rare classes
weights = weights/weights.mean() # weights normalizations
weights = weights.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=weights)

# Model building

In [ ]:
class RecurrentClassifierConv1D(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_layers,
        num_classes,
        rnn_type,
        bidirectional=True,
        dropout_rate=0.2,
        vocab=2,
        embedding_dim=3,
        numerical_feat = len(joint_cols),
        embedded_feat=1,
        pain_feat = 4
    ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

        numerical_feat = len(joint_cols)
        conv_channels = 16 

        #1D convolutional layer
        self.conv = nn.Conv1d(
            in_channels=numerical_feat,   # numero di feature
            out_channels=conv_channels,   # numero di feature convolute
            kernel_size=3,
            padding=1,                    # mantiene stessa lunghezza temporale
            bias=False
        )

        input_size = int(conv_channels + embedded_feat*embedding_dim + pain_feat)

        self.embedding = nn.Embedding(vocab, embedding_dim)
        
        rnn_map = {
            'RNN': nn.RNN,
            'LSTM': nn.LSTM,
            'GRU': nn.GRU
        }

        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', 'GRU'")

        rnn_module = rnn_map[rnn_type]

        dropout_val = dropout_rate if num_layers > 1 else 0

        # Recurrent Layer
        self.rnn = rnn_module(
            input_size =  input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True, # Input shape: (batch, seq_len, features)
            bidirectional = bidirectional,
            dropout = dropout_val
        )

        # Calculate input size for the final classifier
        if self.bidirectional:
            classifier_input_size = hidden_size * 2 # Concat fwd + bwd
        else:
            classifier_input_size = hidden_size

        # Classification layer
        self.classifier = nn.Linear(classifier_input_size, num_classes)

        self.log_softmax = nn.LogSoftmax(dim=1)

    
    def forward(self, x_emb, x_joint, x_pain):

        x_post_emb = self.embedding(x_emb.long())

        x_post_emb = x_post_emb.view(x_post_emb.size(0), x_post_emb.size(1), -1) # flatten the last two dimenions

        #1D convolution
        x_joint_perm = x_joint.permute(0, 2, 1)   # (B, F, T)
        x_conv = self.conv(x_joint_perm)          # (B, conv_channels, T)
        x_conv = x_conv.permute(0, 2, 1)          # (B, T, conv_channels)   

        x = torch.cat((x_conv, x_post_emb, x_pain), dim=-1) # Concatenate the categorical and numerical part
        
        # rnn_out shape: (batch_size, seq_len, hidden_size * num_features)
        rnn_out, hidden = self.rnn(x)    # rnn_out and hidden are the same thing, it's just torch preference to output them both

        # LSTM returns (h_n, c_n), we only need h_n
        if self.rnn_type == 'LSTM':       # h_n = hidden state, c_n = cell state
            hidden = hidden[0]

        # hidden shape: (num_layers * num_features, batch_size, hidden_size)

        if self.bidirectional:
            # Reshape to (num_layers, 2, batch_size, hidden_size)
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)

            # Concat last fwd (hidden[-1, 0, ...]) and bwd (hidden[-1, 1, ...])
            # Final shape: (batch_size, hidden_size * 2)
            hidden_to_classify = torch.cat([hidden[-1, 0, :, :], hidden[-1, 1, :, :]], dim=1)
        else:
            # Take the last layer's hidden state
            # Final shape: (batch_size, hidden_size)
            hidden_to_classify = hidden[-1]

        # Get logits
        logits = self.classifier(hidden_to_classify)
        logits = self.log_softmax(logits)
        return logits

# Model training

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
  
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs_emb, inputs_joint, inputs_pain, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs_emb, inputs_joint, inputs_pain, targets = inputs_emb.to(device).float(), inputs_joint.to(device).float(), inputs_pain.to(device).float(), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs_emb, inputs_joint, inputs_pain)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs_emb.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [ ]:
def validate_one_epoch(model, val_loader, criterion, device):

    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs_emb, inputs_joint, inputs_pain, targets in val_loader:
            # Move data to device
            inputs_emb, inputs_joint, inputs_pain, targets = inputs_emb.to(device).float(), inputs_joint.to(device).float(), inputs_pain.to(device).float(), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs_emb, inputs_joint, inputs_pain)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs_emb.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [ ]:
def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max', restore_best_weights=True, writer=None, verbose=10, experiment_name=""):
    
    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }
    
    # Configure early stopping if patience is set
    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0
    
    print(f"Training {epochs} epochs...")
    
    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):
    
        # Forward pass through training data, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )
    
        # Evaluate model on validation data without updating weights
        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )
    
        # Store metrics for plotting and analysis
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)
    
    
        # Print progress every N epochs or on first epoch
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                    f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                    f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")
    
        # Early stopping logic: monitor metric and save best model
        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)
    
            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered after {epoch} epochs.")
                    break
    
    # Restore best model weights if early stopping was used
    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")
    
    # Save final model if no early stopping
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
    
    return model, training_history

In [ ]:
# Create model and display architecture with parameter count
lstm_conv = RecurrentClassifierConv1D(
    hidden_size=HIDDEN_SIZE,
    num_layers=HIDDEN_LAYERS,
    num_classes=num_classes,
    dropout_rate=DROPOUT_RATE,
    rnn_type='LSTM',
    ).to(device)


# Define optimizer with L2 regularization
optimizer = torch.optim.AdamW(lstm_conv.parameters(), lr=LEARNING_RATE, weight_decay=L2_LAMBDA)

# Enable mixed precision training for GPU acceleration
scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

In [ ]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

In [ ]:
%%time

model_path = '/kaggle/working/models/lstm_con.pth'


# Train model and track training history
lstm_conv, training_history = fit(
    model=lstm_conv,
    train_loader=train_cat_loader,
    val_loader=val_cat_loader,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scaler=scaler,
    device=device,
    verbose=10,
    experiment_name="lstm_emb",
    patience=PATIENCE
    )
    
# Update best model if current performance is superior
if training_history['val_f1'][-1] > best_performance:
    best_model = lstm_conv
    best_performance = training_history['val_f1'][-1]

# Save model
torch.save(lstm_conv.state_dict(), model_path)


In [ ]:
# Create a figure with two side-by-side subplots (two columns)
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 5))

# Plot of training and validation loss on the first axis
ax1.plot(training_history['train_loss'], label='Training loss', alpha=0.3, color='#ff7f0e', linestyle='--')
ax1.plot(training_history['val_loss'], label='Validation loss', alpha=0.9, color='#ff7f0e')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot of training and validation accuracy on the second axis
ax2.plot(training_history['train_f1'], label='Training f1', alpha=0.3, color='#ff7f0e', linestyle='--')
ax2.plot(training_history['val_f1'], label='Validation f1', alpha=0.9, color='#ff7f0e')
ax2.set_title('F1 Score')
ax2.legend()
ax2.grid(alpha=0.3)

# Adjust the layout and display the plot
plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

In [ ]:
# Collect predictions and ground truth labels
val_preds, val_targets = [], []
with torch.no_grad():  # Disable gradient computation for inference
    for xb_emb, xb_joint, xb_pain, yb in val_cat_loader:
        xb_emb = xb_emb.to(device).float()
        xb_joint = xb_joint.to(device).float()
        xb_pain = xb_pain.to(device).float()

        # Forward pass: get model predictions
        logits = lstm_conv(xb_emb, xb_joint, xb_pain)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Store batch results
        val_preds.append(preds)
        val_targets.append(yb.numpy())

# Combine all batches into single arrays
val_preds = np.concatenate(val_preds)
val_targets = np.concatenate(val_targets)

# Calculate overall validation metrics
val_acc = accuracy_score(val_targets, val_preds)
val_prec = precision_score(val_targets, val_preds, average='weighted')
val_rec = recall_score(val_targets, val_preds, average='weighted')
val_f1 = f1_score(val_targets, val_preds, average='weighted')
print(f"Accuracy over the validation set: {val_acc:.4f}")
print(f"Precision over the validation set: {val_prec:.4f}")
print(f"Recall over the validation set: {val_rec:.4f}")
print(f"F1 score over the validation set: {val_f1:.4f}")

# Generate confusion matrix for detailed error analysis
cm = confusion_matrix(val_targets, val_preds)

# Create numeric labels for heatmap annotation
labels = np.array([f"{num}" for num in cm.flatten()]).reshape(cm.shape)

# Visualise confusion matrix
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=labels, fmt='',
            cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

# Evaluation

In [ ]:
# Set model to evaluation mode
lstm_conv.eval()

sample_indices = []
predictions = []
all_preds = []

idx_to_label = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}

majority_voting = {0: 0, 1: 0, 2: 0}

segment_len = X_test_seq_emb.shape[0] // 1324  # frames per sample

frame_counter = 0
sample_count = 0

with torch.no_grad():
    for batch_emb, batch_joint, batch_pain in test_cat_loader:

        xb_emb = batch_emb[0] if isinstance(batch_emb, list) else batch_emb
        xb_joint = batch_joint[0] if isinstance(batch_joint, list) else batch_joint
        xb_pain = batch_pain[0] if isinstance(batch_pain, list) else batch_pain

        xb_emb = xb_emb.to(device).float()
        xb_joint = xb_joint.to(device).float()
        xb_pain = xb_pain.to(device).float()

        logits = lstm_conv(xb_emb, xb_joint, xb_pain)
        preds = logits.argmax(dim=1).cpu().numpy()

        for pred_idx in preds:
            all_preds.append(pred_idx)
            majority_voting[pred_idx] += 1
            frame_counter += 1

            # chiusura sample
            if frame_counter == segment_len:
                majority_class = max(majority_voting, key=majority_voting.get)

                sample_indices.append(f"{sample_count:03d}")
                predictions.append(idx_to_label[majority_class])

                # reset
                majority_voting = {0: 0, 1: 0, 2: 0}
                frame_counter = 0
                sample_count += 1

submission_df = pd.DataFrame({
    'sample_index': sample_indices,
    'label': predictions
})

In [ ]:
submission_df.to_csv('/kaggle/working/submission_conv_stratMajorityConv.csv', index=False)

In [ ]:
print("\nPrediction distribution:")
print(submission_df['label'].value_counts())